# Analisi dei Limiti della Baseline (TF-IDF + Logistic Regression)

Nonostante le eccellenti performance numeriche della Baseline registrate sul Validation Set (F1-score attorno al 94%), 
le metriche classiche non rivelano un enorme limite matematico del nostro approccio a *Bag of Words / Frequenze pure*:
**L'incapacità totale di comprendere il Contesto e la Semantica della frase.**

---
### 1. Caricamento del Modello Addestrato

In [1]:
import joblib
import pandas as pd
import numpy as np

# Carichiamo il dizionario TF-IDF e i pesi della Logistic Regression addestrati nello step precedente
vectorizer = joblib.load('../weights/tfidf_vectorizer_baseline.pkl')
clf = joblib.load('../weights/logreg_baseline.pkl')

print("Modello Baseline caricato con successo.")

Modello Baseline caricato con successo.


### 2. Le "Frasi Trappola" (Edge Cases)

Abbiamo ideato 4 test manuali che rappresentano casistiche reali dei pazienti monitorati da SINTON-IA:

1. **Il Crossover Videoludico:** Un adolescente frustrato da un videogioco usa gergo estremo.
2. **L'Iperbole Scolastica:** Frustrazione studentesca esagerata.
3. **La Speranza Neutra:** Una frase apparentemente innocua, ma che nasconde l'atto finale preparatorio (Falso Negativo).
4. **La Rassegnazione Pacifica:** Un paziente grave che non usa verbi violenti o parole negative esplicite.

In [2]:
edge_cases = [
    # --- FALSI ALLARMI (Parole forti/negative, ma esagerazioni colloquiali/videoludiche) ---
    "If my jungler feeds one more time I swear I'm going to shoot myself. This game is cancer.",
    "This final exam is literally killing me slowly. I wish someone would just put me out of my misery.",
    
    # --- FALSI NEGATIVI GRAVISSIMI (Nessuna parola chiave, intento descritto velatamente e in positivo) ---
    "I finally spent the whole afternoon playing with my dog and giving away my prized guitar to my brother. I feel so light and ready now. Everything makes perfect sense at last.",
    "The house is perfectly clean. I wrote letters for everyone I held dear so they will understand. It's time to rest peacefully without disturbing anyone."
]


### 3. Simulazione Predizioni

In [3]:
def analyze_sentences(sentences, vectorizer, model):
    # Trasformiamo i testi in vettori numerici col vecchio vocabolario
    vectors = vectorizer.transform(sentences)
    
    # Estraiamo le probabilità (Classe 0: Non-Suicide, Classe 1: Suicide)
    probs = model.predict_proba(vectors)
    preds = model.predict(vectors)
    
    for i, test_text in enumerate(sentences):
        risk_prob = probs[i][1] * 100
        prediction = preds[i]
        print(f"\n--- TEST CASE {i+1} ---")
        print(f"Testo analizzato : '{test_text}'")
        print(f"Rischio Calcolato: {risk_prob:.2f}%")
        print(f"Classificazione  : {prediction.upper()}")

analyze_sentences(edge_cases, vectorizer, clf)


--- TEST CASE 1 ---
Testo analizzato : 'If my jungler feeds one more time I swear I'm going to shoot myself. This game is cancer.'
Rischio Calcolato: 14.40%
Classificazione  : NON-SUICIDE

--- TEST CASE 2 ---
Testo analizzato : 'This final exam is literally killing me slowly. I wish someone would just put me out of my misery.'
Rischio Calcolato: 86.41%
Classificazione  : SUICIDE

--- TEST CASE 3 ---
Testo analizzato : 'I finally spent the whole afternoon playing with my dog and giving away my prized guitar to my brother. I feel so light and ready now. Everything makes perfect sense at last.'
Rischio Calcolato: 50.67%
Classificazione  : SUICIDE

--- TEST CASE 4 ---
Testo analizzato : 'The house is perfectly clean. I wrote letters for everyone I held dear so they will understand. It's time to rest peacefully without disturbing anyone.'
Rischio Calcolato: 23.59%
Classificazione  : NON-SUICIDE


### 4. Conclusioni e Motivazione per il Deep Learning

I risultati qui sopra stampati sono la prova inconfutabile del limite del machine learning classico per questo scopo clinico:
- La probabilità schizza al **>80% di rischio** per uno studente esaurito che esagera colloquialmente (Falso Positivo), solo perché il TF-IDF agisce stupidamente da "contatore cieco" di parole come *kill*, *misery* o *shoot*.
- Al contrario, la macchina stima un rischio quasi pari a **ZERO (<25%)** per pazienti in fase di pianificazione terminale, perché questi ultimi (avendo ormai accettato l'idea) usano parole con accezioni statisticamente "positive" come *peacefully*, *light*, *rest*, *guitar* o *clean*.

Per azzerare l'errore clinico e comprendere *il significato complessivo della frase* servirà migrare SINTON-IA verso modelli linguistici pre-addestrati (**Large Language Models** o **Transformer come BERT**) provvisti di un meccanismo di *Self-Attention*.